In [ ]:
from IPython.display import clear_output
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import scanpy as sc
import stlearn as st
st.settings.set_figure_params(dpi=800)
from PIL import Image, ImageChops, ImageDraw
import seaborn as sns
from skimage.color import rgb2hed
from scipy.ndimage.filters import gaussian_filter
import sys
from time import sleep
from tqdm.auto import tqdm
import anndata as ad
import seaborn as sns
from scipy import stats
import sopa

In [ ]:
import subprocess
system = subprocess.check_output(["hostname", "-s"]).decode("utf-8").strip()
BASE_PATH_ = Path()
if "bun" in system:
    BASE_PATH_ = Path("/scratch/project_mnt/S0010/Xiao/Q1851/Xiao/")
elif "imb-quan-gpu" in system:
    BASE_PATH_ = Path("/home/uqxtan9/Q1851/Xiao/")


BASE_PATH = BASE_PATH_ / "Working_project/MB"
DATA_PATH = BASE_PATH / "Xenium_Brain"
XENIUM_RAW_PATH = DATA_PATH / "Xenium_RAW"
MALDI_RAW_PATH = DATA_PATH / "MALDI_RAW/imzml_file"
PROCESSED_ = Path("/home/uqxtan9/Q2051/Xiao/Working_project/MB/PROCESSED")
PROCESSED_.mkdir(exist_ok=True, parents=True)
PROCESSED = BASE_PATH / "PROCESSED"
PROCESSED.mkdir(exist_ok=True, parents=True)
OUT_PATH = Path("/home/uqxtan9/Q2051/Xiao/Working_project/MB/") / "PLOTS" / "Xenium"
OUT_PATH.mkdir(exist_ok=True, parents=True)
QC_PATH = OUT_PATH / "QC"
QC_PATH.mkdir(exist_ok=True, parents=True)
CLS_PATH = OUT_PATH / "CLUSTERING"
CLS_PATH.mkdir(exist_ok=True, parents=True)
CCI_PATH = OUT_PATH / "CCI"
CCI_PATH.mkdir(exist_ok=True, parents=True)

In [ ]:
def annotation_markers(adata, gene_list):
    annotation_dict = dict(gene_list[["Gene", "Primary_cell_types"]].values)
    df = adata.to_df()
    all_annotations = gene_list["Primary_cell_types"].value_counts().index
    df_annotation = pd.DataFrame(index = df.index, columns=all_annotations)
    for index, row in df.iterrows():
        genes = row.index[row>0]
        annotation = genes.map(annotation_dict)
        annotation = pd.Categorical(annotation, categories=all_annotations)
        df_annotation.loc[index,:] = annotation.value_counts() / gene_list["Primary_cell_types"].value_counts()
    return df_annotation

def cell_typing(sample):
    adata = sc.read(PROCESSED / f"{sample}_processed.h5ad")
    gmean_df = adata.to_df().transpose()
    gmean_df = gmean_df.div(gmean_df.max(axis=1), axis=0)
    gmean_df["marker"] = gmean_df.index.map(annotation_dict)
    gmean_df_p = gmean_df.groupby("marker").mean()
    # gmean_df_p = pd.DataFrame(groupd.to_list(), index=groupd.index, columns=gmean_df.columns[:-1])
    gmean_df_p = gmean_df_p.loc[cell_type_list,:]
    gmean_df_p.idxmax().value_counts()
    adata.obs["cell_type"] = gmean_df_p.idxmax()
    adata.obs["cell_type"] = pd.Categorical(adata.obs["cell_type"])
    adata.obs.to_csv(PROCESSED_ / f"{sample}_marker_annotation_tuan.csv")
    adata.write_h5ad(PROCESSED_ / f"{sample}_processed_marker_label_tuan.h5ad")
    

In [ ]:
sample_names = [f.stem for f in XENIUM_RAW_PATH.iterdir() if f.is_dir()]
sample_names

In [ ]:
gene_list = pd.read_csv(BASE_PATH / "Xenium_reannotation.csv")

In [ ]:
gene_list = gene_list.loc[gene_list["Primary_cell_types"].isin(['Astrocytes', 'Neural progenitor cells', 'Neural Stem Cells',
       'Neurons',
       'Microglia', 'Endothelial cells',
       'Oligodendrocytes', 'OPC', 'VLMC',
       'Neural Stem Cells ', 'Proliferation', 'Natural Killer (NK) Cells']),:]
annotation_dict = dict(gene_list[["Gene", "Primary_cell_types"]].values)
cell_type_list = gene_list["Primary_cell_types"].unique()

In [ ]:
for i in sample_names:
    cell_typing(i)